# RFM Customer Segmentation — TheLook eCommerce

**Scope:** Segment customers based on Recency, Frequency, and Monetary value to identify high-value and retention opportunities.

**Input:** `../data/output/raw_rfm.csv` (calculated and exported from `sql/customer_analysis.sql`)

**Output:** `../data/output/rfm_scored.csv` (loaded back into database as `customer_segments`)


## 0. Imports & Load Data

In [1]:
import os
import numpy as np
import pandas as pd

INPUT_PATH = os.path.join("..", "data", "output", "raw_rfm.csv")
OUTPUT_PATH = os.path.join("..", "data", "output", "rfm_scored.csv")

rfm_data = pd.read_csv(INPUT_PATH, index_col='customer_id')
rfm_data.info()

<class 'pandas.DataFrame'>
Index: 27775 entries, 42711 to 44362
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   recency    27775 non-null  int64  
 1   frequency  27775 non-null  int64  
 2   monetary   27775 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 868.0 KB


In [2]:
rfm_data.head()

,recency,frequency,monetary
customer_id,,,
42711,1083,1,67.989998
48284,778,1,29.990000
82244,449,1,8.500000
65264,885,1,25.000000
14324,556,1,5.990000


In [3]:
rfm_data.describe().round(2)

,recency,frequency,monetary
count,27775.00,27775.00,27775.00
mean,460.42,1.13,98.27
std,426.75,0.37,105.23
min,1.00,1.00,0.49
25%,111.00,1.00,32.89
50%,328.00,1.00,63.74
75%,705.00,1.00,128.00
max,1920.00,4.00,1221.62


## 1. RFM Scoring

- **F score (1–3):** Purchase frequency is heavily right-skewed (88% F=1, 11% F=2, ~1% F>=3). Score rule: `F=1 -> 1`, `F=2 -> 2`, `F>=3 -> 3`
- **R score (1–5):** Quintiles via `pd.qcut` with labels `[5, 4, 3, 2, 1]` (most recent customers receive score 5)
- **M score (1–5):** Quintiles via `pd.qcut` with labels `[1, 2, 3, 4, 5]` (highest spenders receive score 5)

In [4]:
def f_score_rule(freq):
    if freq == 1:
        return 1
    elif freq == 2:
        return 2
    else:
        return 3

rfm_data['F_score'] = rfm_data['frequency'].apply(f_score_rule)

rfm_data['R_score'] = pd.qcut(
    rfm_data['recency'],
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm_data['M_score'] = pd.qcut(
    rfm_data['monetary'],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm_data['RFM_score'] = (
    rfm_data['R_score'].astype(str) +
    rfm_data['F_score'].astype(str) +
    rfm_data['M_score'].astype(str)
)

rfm_data.head()

,recency,frequency,monetary,F_score,R_score,M_score,RFM_score
customer_id,,,,,,,
42711,1083,1,67.989998,1,1,3,113
48284,778,1,29.990000,1,2,2,212
82244,449,1,8.500000,1,3,1,311
65264,885,1,25.000000,1,1,1,111
14324,556,1,5.990000,1,2,1,211


In [5]:
rfm_data[['R_score', 'F_score', 'M_score']].describe().round(2)

,R_score,F_score,M_score
count,27775.00,27775.00,27775.00
mean,3.00,1.13,3.00
std,1.41,0.37,1.41
min,1.00,1.00,1.00
25%,2.00,1.00,2.00
50%,3.00,1.00,3.00
75%,4.00,1.00,4.00
max,5.00,3.00,5.00


## 2. Customer Segmentation Logic

> **Recency Context:** Recency thresholds are notably high (P25 ~111 days, P75 ~705 days) because TheLook has a long customer lifecycle and a structural retention gap (88% one-time buyers).

Thresholds are calculated **dynamically from quantiles** rather than hardcoded:
- `p25`: 25th percentile of recency
- `median`: 50th percentile of recency
- `p75`: 75th percentile of recency
- `m_threshold`: 75th percentile of monetary value (top 25% highest spenders)

### Defined Segments:
1. **Loyal Customers:** `frequency >= 3` (high repeat buyers, platform advocates)
2. **Repeat Buyers:** `frequency == 2` (converted to second purchase)
3. **High-Value New:** `frequency == 1`, `recency <= p25`, `monetary >= m_threshold` (recent one-time buyers with top 25% basket size)
4. **New Customers:** `frequency == 1`, `recency <= p25`, `monetary < m_threshold` (standard recent first-time buyers)
5. **Inactive Customers:** `frequency == 1`, `p25 < recency <= p75` (one-time buyers drifting away)
6. **Lost:** `frequency == 1`, `recency > p75` (one-time buyers not seen for over ~2 years)

In [6]:
# Thresholds calculated dynamically from quantiles
p25 = rfm_data['recency'].quantile(0.25)
median = rfm_data['recency'].median()
p75 = rfm_data['recency'].quantile(0.75)
m_threshold = rfm_data['monetary'].quantile(0.75)

print(f"Recency thresholds:  P25 = {p25:.1f} days | Median = {median:.1f} days | P75 = {p75:.1f} days")
print(f"Monetary threshold:  P75 = ${m_threshold:.2f}")

def assign_segment(data):
    if data['frequency'] >= 3:
        return 'Loyal Customers'
    elif data['frequency'] == 2:
        return 'Repeat Buyers'
    elif data['frequency'] == 1:
        if data['recency'] <= p25:
            if data['monetary'] >= m_threshold:
                return 'High-Value New'
            else:
                return 'New Customers'
        elif data['recency'] <= p75:
            return 'Inactive Customers'
        elif data['recency'] > p75:
            return 'Lost'
    return 'Other'

rfm_data['segment'] = rfm_data.apply(assign_segment, axis=1)

Recency thresholds:  P25 = 111.0 days | Median = 328.0 days | P75 = 705.0 days
Monetary threshold:  P75 = $128.00


In [7]:
print("=== Segment Distribution (Counts) ===")
print(rfm_data['segment'].value_counts())

print("\n=== Segment Distribution (%) ===")
print((rfm_data['segment'].value_counts(normalize=True) * 100).round(2))

=== Segment Distribution (Counts) ===
segment
Inactive Customers    12152
Lost                   6529
New Customers          4571
Repeat Buyers          2982
High-Value New         1215
Loyal Customers         326
Name: count, dtype: int64

=== Segment Distribution (%) ===
segment
Inactive Customers    43.75
Lost                  23.51
New Customers         16.46
Repeat Buyers         10.74
High-Value New         4.37
Loyal Customers        1.17
Name: proportion, dtype: float64


In [8]:
# Aggregated segment performance summary
segment_summary = rfm_data.groupby('segment').agg(
    num_customers=('recency', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
    total_monetary=('monetary', 'sum')
).round(2)

segment_summary['pct_customers'] = (
    segment_summary['num_customers'] / segment_summary['num_customers'].sum() * 100
).round(2)

segment_summary['pct_revenue'] = (
    segment_summary['total_monetary'] / segment_summary['total_monetary'].sum() * 100
).round(2)

segment_summary = segment_summary.sort_values(by='total_monetary', ascending=False)
segment_summary

,num_customers,avg_recency,avg_frequency,avg_monetary,total_monetary,pct_customers,pct_revenue
segment,,,,,,,
Inactive Customers,12152,358.82,1.00,86.85,1055364.88,43.75,38.67
Lost,6529,1093.08,1.00,89.25,582694.47,23.51,21.35
Repeat Buyers,2982,321.24,2.00,172.49,514367.56,10.74,18.85
High-Value New,1215,44.83,1.00,219.89,267160.50,4.37,9.79
New Customers,4571,44.10,1.00,49.90,228089.30,16.46,8.36
Loyal Customers,326,235.85,3.06,250.57,81686.77,1.17,2.99


In [9]:
rfm_data.info()

<class 'pandas.DataFrame'>
Index: 27775 entries, 42711 to 44362
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   recency    27775 non-null  int64  
 1   frequency  27775 non-null  int64  
 2   monetary   27775 non-null  float64
 3   F_score    27775 non-null  int64  
 4   R_score    27775 non-null  int64  
 5   M_score    27775 non-null  int64  
 6   RFM_score  27775 non-null  str    
 7   segment    27775 non-null  str    
dtypes: float64(1), int64(5), str(2)
memory usage: 2.3 MB


## 3. Export Scored Segments

In [10]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
rfm_data.to_csv(OUTPUT_PATH)
print(f"✅ Successfully exported {len(rfm_data):,} records to {OUTPUT_PATH}")

✅ Successfully exported 27,775 records to ..\data\output\rfm_scored.csv
